Connecting Database

In [ ]:
import os, getpass
import mysql.connector
import pandas as pd

mydb = mysql.connector.connect(
    host="localhost",
    user="root",
    mysql_password=os.getenv("MYSQL_PASSWORD") or getpass.getpass("MySQL password: "),
    ssl_disabled=True
)
mycursor = mydb.cursor()
print("Connected:", mydb.is_connected())

Connected: True


Creating Database and Table

In [ ]:
mycursor.execute("CREATE DATABASE IF NOT EXISTS police_logs")
mycursor.execute("USE police_logs")
print("Database created or already exists!")

mycursor.execute("DROP TABLE IF EXISTS traffic_stops")
mycursor.execute("""
CREATE TABLE traffic_stops (
    id INT AUTO_INCREMENT PRIMARY KEY,
    stop_date DATE,
    stop_time TIME,
    country_name VARCHAR(100),
    driver_gender VARCHAR(10),
    driver_age INT,
    driver_race VARCHAR(50),
    violation VARCHAR(100),
    search_conducted BOOLEAN,
    search_type VARCHAR(100),
    stop_outcome VARCHAR(50),
    is_arrested BOOLEAN,
    stop_duration VARCHAR(20),
    drugs_related_stop BOOLEAN,
    vehicle_number VARCHAR(20),
    INDEX idx_vehicle (vehicle_number),
    INDEX idx_stop_date (stop_date),
    INDEX idx_country_violation (country_name, violation),
    INDEX idx_outcome (stop_outcome)
)
""")
print("Table 'traffic_stops' created successfully!")

Database created or already exists!
Table 'traffic_stops' created successfully!


Inserting Data

In [3]:
df = pd.read_csv("cleaned_traffic_stops.csv")

# Same column order as the INSERT below
columns = ["stop_date", "stop_time", "country_name", "driver_gender", "driver_age",
           "driver_race", "violation", "search_conducted", "search_type",
           "stop_outcome", "is_arrested", "stop_duration", "drugs_related_stop",
           "vehicle_number"]
df = df[columns]

# Blank search_type cells are NaN in pandas; MySQL needs None (NULL) instead
df = df.astype(object).where(df.notna(), None)

insert_query = """
INSERT INTO traffic_stops (
    stop_date, stop_time, country_name, driver_gender, driver_age,
    driver_race, violation, search_conducted, search_type,
    stop_outcome, is_arrested, stop_duration, drugs_related_stop, vehicle_number
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

rows = list(df.itertuples(index=False, name=None))

# Insert in batches of 5,000 rows (much faster than one row at a time)
for start in range(0, len(rows), 5000):
    mycursor.executemany(insert_query, rows[start:start + 5000])

mydb.commit()
print(f"{len(rows):,} rows inserted successfully!")

C:\Users\yogap\AppData\Local\Temp\ipykernel_19536\29201257.py:1: DtypeWarning: Columns (0: search_type) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("cleaned_traffic_stops.csv")


65,538 rows inserted successfully!


Checking the data

In [4]:
mycursor.execute("SELECT COUNT(*) FROM traffic_stops")
print("Rows in table:", mycursor.fetchone()[0])

mycursor.execute("SELECT * FROM traffic_stops LIMIT 5")
for row in mycursor.fetchall():
    print(row)

Rows in table: 65538
(1, datetime.date(2020, 1, 1), datetime.timedelta(0), 'Canada', 'M', 19, 'Asian', 'Speeding', 1, 'Vehicle Search', 'Ticket', 1, '16-30 Min', 1, 'UP76DY3473')
(2, datetime.date(2020, 1, 1), datetime.timedelta(seconds=60), 'India', 'M', 58, 'Other', 'Other', 0, 'Vehicle Search', 'Arrest', 1, '16-30 Min', 1, 'RJ83PZ4441')
(3, datetime.date(2020, 1, 1), datetime.timedelta(seconds=120), 'USA', 'M', 76, 'Black', 'Speeding', 0, 'Frisk', 'Ticket', 1, '16-30 Min', 1, 'RJ32OM7264')
(4, datetime.date(2020, 1, 1), datetime.timedelta(seconds=180), 'Canada', 'M', 76, 'Black', 'DUI', 1, 'Frisk', 'Warning', 0, '0-15 Min', 1, 'RJ76TI3807')
(5, datetime.date(2020, 1, 1), datetime.timedelta(seconds=240), 'Canada', 'M', 75, 'Other', 'Other', 0, 'Vehicle Search', 'Arrest', 1, '16-30 Min', 0, 'WB63BB8305')


In [ ]:
mycursor.close()
mydb.close()